In [24]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load the Flan model and tokenizer
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generate_prompt(task_type, input_text, additional_info=None):
    """
    Generates a prompt for different NLP tasks.

    Parameters:
        task_type (str): The type of NLP task (e.g., "summarization", "classification").
        input_text (str): The input text for the task.
        additional_info (dict, optional): Additional details like categories or constraints.

    Returns:
        str: The formatted prompt.
    """
    if task_type == "summarization":
        return f"Summarize this text: {input_text}"
    elif task_type == "classification":
        categories = ", ".join(additional_info['categories'])
        return f"Classify this text into one of the following categories: {categories}.\n\nText: {input_text}"
    elif task_type == "translation":
        source = additional_info['source_language']
        target = additional_info['target_language']
        return f"Translate this text from {source} to {target}: {input_text}"
    elif task_type == "sentiment_analysis":
        return f"Analyze the sentiment of this text: {input_text}"
    elif task_type == "text_generation":
        return f"Generate a continuation for this text: {input_text}"
    elif task_type == "ner":
        return f"Extract named entities such as people, places, and organizations from this text: {input_text}"
    else:
        raise ValueError("Unsupported task type!")

def perform_nlp_task(task_type, input_text, additional_info=None, max_length=128):
    """
    Performs an NLP task using the Flan model.

    Parameters:
        task_type (str): The type of NLP task.
        input_text (str): The input text for the task.
        additional_info (dict, optional): Additional task-specific details.
        max_length (int): Maximum length of the output text.

    Returns:
        str: The output generated by the Flan model.
    """
    # Generate the prompt
    prompt = generate_prompt(task_type, input_text, additional_info)

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)

    # Generate the output
    outputs = model.generate(inputs.input_ids, max_length=max_length, num_beams=4, early_stopping=True)

    # Decode the output
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

# Example usage


In [8]:
task = "summarization"
text = ("Kanchipuram silk weaving, a 400-year-old traditional craft", "is famous all over the world for its bright colours, beautiful designs, and pure mulberry silk.",
        "Weaved traditionally for weddings and festivities,", "these sarees are Tamil Nadu's cultural heritage. But this renowned art is in danger now.")
additional_details = {"length": "3 sentences"}

result = perform_nlp_task(task, text, additional_details)
print("Result:", result)

Result: 'Kanchipuram silk weaving, a 400-year-old traditional craft, is famous all over the world for its bright colours, beautiful designs, and pure mulberry silk. But this renowned art is in danger now.'


In [10]:
task = "classification"
text = "The product was not excellent and dispointed my expectations!"
categories = ["Positive", "Negative", "Neutral"]
result = perform_nlp_task(task, text, {"categories": categories})
print(f"Classification Result:\n{result}\n")

Classification Result:
Negative



In [13]:
task = "translation"
text = "Social sustainability involves ensuring equitable access to resources."
result = perform_nlp_task(task, text, {"source_language": "English", "target_language": "French"})
print(f"Translation Result:\n{result}\n")

Translation Result:
La durabilité sociale implique assurer l'accès équitable aux ressources.



In [20]:
task = "sentiment_analysis"
text = "I am neutral on how this initiative has transformed the community!"
result = perform_nlp_task(task, text)
print(f"Sentiment Analysis Result:\n{result}\n")

Sentiment Analysis Result:
positive



In [22]:
task = "text_generation"
text = "The weaving community in Kanchipuram has a rich history. Today, they are focused on....."
result = perform_nlp_task(task, text)
print(f"Text Generation Result:\n{result}\n")

Text Generation Result:
The looms used in the weaving process are made from a variety of materials, such as cotton, silk, and jute.



In [36]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import spacy

# Load Flan-T5 model and tokenizer
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Load spaCy's pre-trained model for NER
nlp = spacy.load("en_core_web_sm")

def generate_prompt(task_type, input_text, additional_info=None):
    """
    Generates a prompt for Named Entity Recognition (NER).
    """
    if task_type == "ner":
        return (
            "Extract all named entities (e.g., people, places, dates, organizations) "
            f"from the following text: {input_text}"
        )
    else:
        raise ValueError(f"Unsupported task type: {task_type}")

def perform_nlp_task(task_type, input_text, max_length=128):
    """
    Performs an NLP task using the Flan model.
    """
    # Generate the prompt
    prompt = generate_prompt(task_type, input_text)

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)

    # Generate the output
    outputs = model.generate(inputs.input_ids, max_length=max_length, num_beams=4, early_stopping=True)

    # Decode the output
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result

def extract_entities_with_spacy(input_text):
    """
    Extracts named entities using spaCy for post-processing.
    """
    doc = nlp(input_text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]
    return entities

# Example usage
if __name__ == "__main__":
    # Input text
    text = "John visited India on April 5, 2025, and met with Dr. Maria to discuss community health."

    # Flan NER result
    task = "ner"
    flan_result = perform_nlp_task(task, text, max_length=256)
    print(f"Flan NER Result:\n{flan_result}\n")

    # spaCy NER result
    spacy_result = extract_entities_with_spacy(text)
    print("spaCy NER Result:")
    for entity, label in spacy_result:
        print(f"  {entity} ({label})")


Flan NER Result:
John

spaCy NER Result:
  John (PERSON)
  India (GPE)
  April 5, 2025 (DATE)
  Maria (PERSON)


RAG

In [2]:
!pip install faiss-cpu transformers datasets torch

In [5]:
import faiss
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset
import torch

# Load the Flan model and tokenizer
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def build_faiss_index(corpus, model_name="google/flan-t5-large"):
    """
    Builds a FAISS index from a text corpus.

    Parameters:
        corpus (list): A list of text documents.
        model_name (str): The model name for tokenizer embeddings.

    Returns:
        index, corpus: The FAISS index and original corpus for reference.
    """
    # Tokenizer and embedding model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # Tokenize and encode corpus
    inputs = tokenizer(corpus, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        embeddings = model.encoder(input_ids=inputs.input_ids).last_hidden_state.mean(dim=1).numpy()

    # Build FAISS index
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    return index, corpus

def retrieve_documents(query, index, corpus, top_k=3, model_name="google/flan-t5-large"):
    """
    Retrieves top-k documents from FAISS index for a query.

    Parameters:
        query (str): The input query.
        index: The FAISS index.
        corpus (list): The original corpus.
        top_k (int): Number of documents to retrieve.

    Returns:
        list: Top-k retrieved documents.
    """
    # Tokenizer and embedding model
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    # Tokenize and encode query
    inputs = tokenizer(query, return_tensors="pt", padding=True, truncation=True, max_length=512)
    with torch.no_grad():
        query_embedding = model.encoder(input_ids=inputs.input_ids).last_hidden_state.mean(dim=1).numpy()

    # Retrieve documents
    distances, indices = index.search(query_embedding, top_k)
    return [corpus[idx] for idx in indices[0]]

def generate_response(query, retrieved_docs, model_name="google/flan-t5-large"):
    """
    Generates a response using the query and retrieved documents.

    # Combine query and retrieved documents
    context = " ".join(retrieved_docs)
    input_text = f"Query: {query}\nContext: {context}\nAnswer:"

    # Tokenize and generate
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)
    outputs = model.generate(inputs.input_ids, max_length=128, num_beams=4, early_stopping=True)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Example Usage
if __name__ == "__main__":
    # Corpus of knowledge documents
    corpus = [
 "Patient presented with fever and cough for the past three days. The cough is primarily dry, and the patient reports occasional shortness of breath. No significant weight loss has been noted.",  # Report 1
 "Medical history includes hypertension controlled with medication, a previous episode of pneumonia, and seasonal allergies. The patient takes lisinopril and loratadine regularly. No known drug allergies.",  # Report 2
  "Physical examination revealed a temperature of 102°F, elevated heart rate of 110 bpm, and clear lungs upon auscultation. Mild wheezing was noted during forced expiration. The patient also exhibited mild dehydration.",  # Report 3
   "Lab results showed elevated white blood cell count (WBC 15,000/mm³), with a predominance of neutrophils. Chest X-ray findings were consistent with bronchitis, and no signs of consolidation or pleural effusion were noted.",           # Report 4
    "Treatment plan includes antibiotics such as azithromycin for the suspected bacterial infection, along with a prescription for a cough suppressant and instructions for increased fluid intake. The patient is advised to follow up in one week or sooner if symptoms worsen.",    # Report 5
]

    # Build FAISS index
    index, corpus = build_faiss_index(corpus)

    # Query
    query = "write in brief about the diagnosis in 2-3 sentence"

    # Retrieve documents
    retrieved_docs = retrieve_documents(query, index, corpus)
    print(f"Retrieved Documents: {retrieved_docs}\n")

    # Generate response
    response = generate_response(query, retrieved_docs)
    print(f"Generated Response:\n{response}")


Retrieved Documents: ['Lab results showed elevated white blood cell count (WBC 15,000/mm³), with a predominance of neutrophils. Chest X-ray findings were consistent with bronchitis, and no signs of consolidation or pleural effusion were noted.', 'Treatment plan includes antibiotics such as azithromycin for the suspected bacterial infection, along with a prescription for a cough suppressant and instructions for increased fluid intake. The patient is advised to follow up in one week or sooner if symptoms worsen.', 'Physical examination revealed a temperature of 102°F, elevated heart rate of 110 bpm, and clear lungs upon auscultation. Mild wheezing was noted during forced expiration. The patient also exhibited mild dehydration.']

Generated Response:
Chest X-ray findings were consistent with bronchitis.


PEFT

In [6]:
!pip install transformers peft

In [14]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

In [15]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

In [16]:
model_name = "google/flan-t5-large"  # Replace with your desired model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [17]:
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,  # Adjust for your task
    r=8,  # Rank of the LoRA update matrices
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.05,  # Dropout probability
    target_modules=["q", "v"],  # Layer to apply LoRA to
)

In [18]:
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()  # Check the number of trainable parameters

trainable params: 2,359,296 || all params: 785,509,376 || trainable%: 0.3004


In [21]:
import time
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import get_peft_model, LoraConfig, TaskType

# Function to measure inference time
def measure_inference_time(model, tokenizer, text):
    start_time = time.time()
    inputs = tokenizer(text, return_tensors="pt")
    outputs = model.generate(**inputs)
    end_time = time.time()
    return end_time - start_time

# Load the base model and tokenizer
model_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Configure and load the PEFT model (using LoRA)
peft_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,  # Task type
    r=8,  # Rank of the LoRA update matrices
    lora_alpha=32,  # Scaling factor
    lora_dropout=0.05,  # Dropout probability
    target_modules=["q", "v"],  # Attention layers
)

# Apply PEFT configuration to the base model
peft_model = get_peft_model(base_model, peft_config)

# Example text for inference
text = "Translate this sentence into French: Hello, how are you?"

# Measure inference time for both models
base_model_time = measure_inference_time(base_model, tokenizer, text)
peft_model_time = measure_inference_time(peft_model, tokenizer, text)

# Compare inference times
print(f"Base Model Inference Time: {base_model_time:.4f} seconds")
print(f"PEFT Model Inference Time: {peft_model_time:.4f} seconds")


Base Model Inference Time: 4.6322 seconds
PEFT Model Inference Time: 3.4318 seconds


EVALUATION

In [26]:
!pip install rouge-score

  Using cached rouge_score-0.1.2.tar.gz (17 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=b3a2594b72e9799a7324f76d56f6e4e4782d2100282609d4f724b44e041b740f
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [30]:
!pip install nltk
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [31]:
# Install required libraries before running
# pip install nltk rouge-score transformers google-api-python-client sentence-transformers datasets

import nltk
from rouge_score import rouge_scorer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from sklearn.metrics.pairwise import cosine_similarity
from googleapiclient import discovery
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import numpy as np

# Download NLTK data (only needs to be run once)
nltk.download('punkt')

### Evaluation Functions

# BLEU Score
def calculate_bleu(reference, candidate):
    from nltk.translate.bleu_score import sentence_bleu
    reference_tokens = [nltk.word_tokenize(reference)]
    candidate_tokens = nltk.word_tokenize(candidate)
    return sentence_bleu(reference_tokens, candidate_tokens)

# ROUGE Score
def calculate_rouge(reference, candidate):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
    scores = scorer.score(reference, candidate)
    return scores

# Toxicity (Perspective API)
def check_toxicity(text, api_key):
    service = discovery.build("commentanalyzer", "v1alpha1", developerKey=api_key)
    analyze_request = {
        'comment': {'text': text},
        'requestedAttributes': {'TOXICITY': {}}
    }
    response = service.comments().analyze(body=analyze_request).execute()
    return response['attributeScores']['TOXICITY']['summaryScore']['value']

# Relevance (Cosine Similarity)
def calculate_relevance(query, response, model_name="sentence-transformers/all-MiniLM-L6-v2"):
    model = SentenceTransformer(model_name)
    query_embedding = model.encode([query])
    response_embedding = model.encode([response])
    similarity = cosine_similarity(query_embedding, response_embedding)[0][0]
    return similarity

# Perplexity (Fluency)
def calculate_perplexity(text, model_name="gpt2"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    tokens = tokenizer(text, return_tensors="pt")
    loss = model(**tokens, labels=tokens["input_ids"]).loss
    perplexity = np.exp(loss.item())
    return perplexity

# GLUE/SuperGLUE Evaluation
def evaluate_glue_task(task_name, model_name="bert-base-uncased"):
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
    dataset = load_dataset("glue", task_name)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    # Tokenize data
    def preprocess_function(examples):
        return tokenizer(examples['sentence1'], examples['sentence2'], truncation=True)

    tokenized_datasets = dataset.map(preprocess_function, batched=True)
    trainer = Trainer(
        model=model,
        args=TrainingArguments(output_dir="./results", evaluation_strategy="epoch"),
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["validation"],
    )
    eval_results = trainer.evaluate()
    return eval_results

# HHH (Helpful, Honest, Harmless)
def evaluate_hhh(response, context=None):
    # Simplified; requires human judgment for deeper insights.
    helpful = "Does the response address the query effectively?"
    honest = "Does the response align with known facts?"
    harmless = "Is the response free of offensive or harmful content?"
    return {"helpful": helpful, "honest": honest, "harmless": harmless}

### Example Usage

if __name__ == "__main__":
    # Example data
    reference_text = "The cat sat on the mat."
    candidate_text = "The cat was sitting on the mat."
    query_text = "What is the capital of France?"
    response_text = "The capital of France is Paris."

    # BLEU
    bleu_score = calculate_bleu(reference_text, candidate_text)
    print(f"BLEU Score: {bleu_score}")

    # ROUGE
    rouge_scores = calculate_rouge(reference_text, candidate_text)
    print(f"ROUGE Scores: {rouge_scores}")

    # Toxicity (requires API key)
    # api_key = "YOUR_PERSPECTIVE_API_KEY"
    # toxicity_score = check_toxicity(response_text, api_key)
    # print(f"Toxicity Score: {toxicity_score}")

    # Relevance (Cosine Similarity)
    relevance_score = calculate_relevance(query_text, response_text)
    print(f"Relevance Score (Cosine Similarity): {relevance_score}")

    # Perplexity (Fluency)
    perplexity_score = calculate_perplexity(response_text)
    print(f"Perplexity Score: {perplexity_score}")

    # GLUE Task (e.g., "sst2" for sentiment analysis)
    # glue_results = evaluate_glue_task("sst2")
    # print(f"GLUE Task Results: {glue_results}")

    # HHH
    hhh_scores = evaluate_hhh(response_text)
    print(f"HHH Evaluation: {hhh_scores}")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


BLEU Score: 0.4111336169005197
ROUGE Scores: {'rouge1': Score(precision=0.7142857142857143, recall=0.8333333333333334, fmeasure=0.7692307692307692), 'rouge2': Score(precision=0.5, recall=0.6, fmeasure=0.5454545454545454), 'rougeL': Score(precision=0.7142857142857143, recall=0.8333333333333334, fmeasure=0.7692307692307692)}


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Perplexity Score: 44.62388580578848
HHH Evaluation: {'helpful': 'Does the response address the query effectively?', 'honest': 'Does the response align with known facts?', 'harmless': 'Is the response free of offensive or harmful content?'}


In [32]:
from transformers import pipeline
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# Load sentence transformer model for relevance
relevance_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Load toxicity detection pipeline
toxicity_classifier = pipeline("text-classification", model="unitary/toxic-bert")

def evaluate_hhh(query, response, reference=None):
    # 1. Helpful: Cosine similarity and ROUGE (if reference provided)
    helpfulness = {
        "cosine_similarity": calculate_relevance(query, response),
        "rouge_recall": calculate_rouge(reference, response)["rougeL"].recall if reference else None
    }

    # 2. Honest: Placeholder for factual consistency model or manual fact-check
    honesty = "Manual or fact-check model needed"  # Integration required

    # 3. Harmless: Toxicity score
    toxicity_score = toxicity_classifier(response)[0]["score"]

    harmlessness = {"toxicity_score": toxicity_score, "safe": toxicity_score < 0.3}  # Safe if below 0.3 threshold

    return {"helpful": helpfulness, "honest": honesty, "harmless": harmlessness}

# Example query and response
query_text = "What is the capital of France?"
response_text = "The capital of France is Paris."

hhh_scores = evaluate_hhh(query_text, response_text, reference="The capital of France is Paris.")
print("HHH Evaluation:", hhh_scores)


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Device set to use cuda:0


HHH Evaluation: {'helpful': {'cosine_similarity': np.float32(0.87901086), 'rouge_recall': 1.0}, 'honest': 'Manual or fact-check model needed', 'harmless': {'toxicity_score': 0.0006358023383654654, 'safe': True}}
